# DSA 2050 | Data Science Methodology
## Week 2 Practical Lab: SQL, Data Acquisition and Join Validation


**Tools:** Jupyter Notebook, Python, SQLite, pandas  

**Student:** *VICTOR APAMO*  |  **Student ID:** *673804*

> **Core idea:** A SQL query can run successfully and still produce the wrong analytical result. Always check keys, row counts, grain, unmatched records and totals after joins.



---
# Part A: Create the Source Data
Create three small source datasets: `customers.csv`, `orders.csv` and `regions.json`.

### Step 1: Import Libraries

In [1]:
import pandas as pd
import sqlite3
import json

### Step 2: Create the Customer Data

> **Observe:** One `CustomerID` appears more than once (`C004`). Leave it in place for now. It is investigated in Part C.

In [2]:
customers = pd.DataFrame({
    "CustomerID": [
        "C001", "C002", "C003", "C004",
        "C004", "C005", "C006", "C007",
        "C008", "C009", "C010"
    ],
    "CustomerName": [
        "Amina", "Brian", "Carol", "David",
        "David", "Esther", "Faith", "George",
        "Halima", "Ian", "Jane"
    ],
    "Segment": [
        "Retail", "Corporate", "Retail", "SME",
        "SME", "Corporate", "SME", "Retail",
        "Corporate", "Retail", "SME"
    ],
    "RegionCode": [
        "NRB", "MSA", "NRB", "KSM",
        "KSM", "NKR", "NRB", "MSA",
        "KSM", "NKR", "NRB"
    ]
})
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
customers.to_csv("customers.csv", index=False)

### Step 3: Create the Orders Data

In [4]:
orders = pd.DataFrame({
    "OrderID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015"
    ],
    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],
    "OrderAmount": [
        4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],
    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Office",
        "Electronics", "Home", "Office", "Home", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
    ]
})
orders

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [5]:
orders.to_csv("orders.csv", index=False)

### Step 4: Create the JSON Region Lookup

In [6]:
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]

with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

In [7]:
import os
print([f for f in ["customers.csv", "orders.csv", "regions.json"] if os.path.exists(f)])

['customers.csv', 'orders.csv', 'regions.json']


---
# Part B: Inspect the Sources `[2 Marks]`
Reload the CSV files so that the analysis begins from the saved source files rather than the in-memory DataFrames.

In [8]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

In [9]:
display(customers.head())
display(orders.head())

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


In [10]:
print("customers:", customers.shape)
print("orders   :", orders.shape)

customers: (11, 4)
orders   : (15, 4)


> **Key concept, Grain:** The grain describes what one row represents. A customer table and an order table usually have different grains, even when both contain `CustomerID`.

### Q1. What does one row represent in `customers.csv`?
**Answer:** One row represents one customer record: a customer's ID, name, segment and region code. The intended grain is *one row per customer*. (In practice `C004` breaks this because it appears twice, see Part C.)

### Q2. What does one row represent in `orders.csv`?
**Answer:** One row represents one order (a single transaction), identified by `OrderID`, with the customer who placed it, the order amount and the product category. The grain is *one row per order*.

### Q3. Are the two datasets at the same aggregation grain? Explain briefly.
**Answer:** No. `customers.csv` is at customer grain (about 10 distinct customers) and `orders.csv` is at order grain (15 orders). One customer can place many orders (for example `C001`, `C002`, `C003` and `C006` each have two), so the relationship is **one customer to many orders**. This is why `CustomerID` must be unique in the customer table before joining.

---
# Part C: Detect Key Problems `[3 Marks]`
Before joining relational data, verify that the key behaves as expected.

### Step 1: Check Duplicate Customer IDs

In [11]:
customers["CustomerID"].duplicated().sum()

np.int64(1)

In [12]:
customers[customers["CustomerID"].duplicated(keep=False)]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


**Finding:** There is **1** duplicated `CustomerID` row: `C004` (David, SME, KSM) appears twice with identical details.

### Q4. Why can a duplicate `CustomerID` be dangerous when joining customer and order tables?
**Answer:** In a join, every order is matched against *every* customer row with the same key. If `C004` appears twice in the customer table, each `C004` order matches two rows, so it is returned twice. The result has more rows than orders, and any `SUM` or `COUNT` is inflated. The query still runs without an error, so the wrong totals can go unnoticed and lead to wrong business decisions.

### Step 2: Check Unmatched Customer IDs

In [13]:
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

### Q5. Which `CustomerID` appears in orders but not in customers?
**Answer:** `C999` (order `O015`, KSh 9,900, Electronics).

### Q6. Why should an unmatched record be investigated rather than silently deleted?
**Answer:** An unmatched order is real revenue (KSh 9,900), so deleting it would understate sales. It may point to a root cause: a customer missing from the customer master, a typo in the ID, a customer created in another system, or a data-entry or integration error. Investigating and documenting it lets the business fix the source problem and decide how to treat the order. Silent deletion hides the issue and makes the numbers unreproducible.



---
# Part D: Create the SQLite Database `[2 Marks]`
SQLite lets us practise relational querying locally without a separate database server.

In [14]:
conn = sqlite3.connect("retail_lab.db")

In [15]:
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

15

### Test the Database

In [16]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


#### Verification checklist

In [17]:
print("retail_lab.db exists:", os.path.exists("retail_lab.db"))
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn))
print("Rows returned by test query:", len(pd.read_sql_query("SELECT * FROM customers LIMIT 5;", conn)))

retail_lab.db exists: True
        name
0  customers
1     orders
Rows returned by test query: 5


- [x] The query returns five customer rows.
- [x] The SQLite file `retail_lab.db` exists in the project folder.
- [x] Both `customers` and `orders` tables were created successfully.

---
# Part E: SELECT and WHERE `[4 Marks]`
Use SQL to retrieve only the data needed for a business question.

### Task E1: Basic SELECT
Write a query that returns only `OrderID`, `CustomerID` and `OrderAmount`.

In [18]:
query = """
SELECT
    OrderID,
    CustomerID,
    OrderAmount
FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


### Task E2: Filter with WHERE
> **Hint:** Combine multiple conditions using `AND`. Put text values inside single quotes in SQL.

**E2 (a)** Write a query that returns only customers in the **Retail** segment.

In [19]:
query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


**E2 (b)** Write a query that returns orders with `OrderAmount` **greater than KSh 8,000**.

In [20]:
query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


**E2 (c)** Write a query that returns only **Electronics** orders.

In [21]:
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


**E2 (d)** Write a query that returns **Electronics orders worth more than KSh 8,000**.

In [22]:
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
  AND OrderAmount > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


---
# Part F: GROUP BY and Aggregate Functions `[3 Marks]`
Management usually needs summarised information rather than raw transaction rows. Use SQL aggregates to calculate category-level KPIs.

### Step 1: Total Sales

In [23]:
query = """
SELECT
    SUM(OrderAmount) AS TotalSales
FROM orders;
"""
pd.read_sql_query(query, conn)

,TotalSales
0,113500


### Step 2: Sales by Product Category

In [24]:
query = """
SELECT
    ProductCategory,
    SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


### Your Task
Create **one** SQL query that returns all three KPIs for each product category: number of orders (`COUNT()`), total sales (`SUM()`) and average order value (`AVG()`).

In [25]:
query = """
SELECT
    ProductCategory,
    COUNT(*)                     AS NumberOfOrders,
    SUM(OrderAmount)             AS TotalSales,
    ROUND(AVG(OrderAmount), 2)   AS AverageOrderValue
FROM orders
GROUP BY ProductCategory
ORDER BY TotalSales DESC;
"""
category_kpis = pd.read_sql_query(query, conn)
category_kpis

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.33
1,Office,5,42500,8500.00
2,Home,4,19500,4875.00


| Product Category | Number of Orders | Total Sales (KSh) | Average Order Value (KSh) |
|---|---:|---:|---:|
| Electronics | 6 | 51,500 | 8,583.33 |
| Home | 4 | 19,500 | 4,875.00 |
| Office | 5 | 42,500 | 8,500.00 |

### Q7. Write two short observations about category performance. Refer to actual values from your output.
**Answer:**
1. **Electronics is the top category** with KSh 51,500 in sales from 6 orders, about 45% of the KSh 113,500 total. It has both the most orders and the highest average order value (KSh 8,583.33).
2. **Home is the weakest category**, with only 4 orders, KSh 19,500 total and an average order value of KSh 4,875, which is roughly 43% lower than Electronics (8,583.33). Office is close to Electronics on order value (KSh 8,500) but has one order fewer, so Office orders are about as valuable as Electronics orders, while Home orders are much smaller.

---
# Part G: JOIN Customers and Orders `[3 Marks]`
Connect order transactions to customer information. Use a `LEFT JOIN` so that every order remains visible, including orders whose customer record is missing.

In [26]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
    ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [27]:
print("Rows in orders :", len(orders))
print("Rows in joined :", len(joined))

Rows in orders : 15
Rows in joined : 16


In [28]:
# Inspect exactly which orders were affected
joined[joined["OrderID"].duplicated(keep=False)]

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100


In [29]:
# Locate the unmatched customer record
joined[joined["CustomerName"].isna()]

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
15,O015,C999,NaN,NaN,9900


> **Think before moving on:** A LEFT JOIN preserves left-side rows, but a non-unique key on the right can multiply rows. "The JOIN worked" is not the same as "the JOIN is correct."

### Q8. Are the row counts equal? If not, explain exactly what caused the difference.
**Answer:** No. `orders` has **15** rows but `joined` has **16**. The extra row is caused by the duplicate customer key `C004`: the customer table has two `C004` rows, so order `O005` (which belongs to `C004`) matches both and appears twice in the join result. A LEFT JOIN never drops left-side rows, but it duplicates them when the right-side key is not unique.

### Q9. Locate the unmatched customer record in the joined output. What values appear for `CustomerName` and `Segment`?
**Answer:** Order `O015` (`CustomerID = C999`, KSh 9,900) is the unmatched record. Because no customer `C999` exists, the LEFT JOIN keeps the order but fills `CustomerName` and `Segment` with **NULL** (shown as `None` / `NaN` in pandas).

---
# Part H: Reconcile the Totals
A correct data integration process should preserve totals unless there is a deliberate reason for them to change.

### Before the JOIN

In [30]:
orders["OrderAmount"].sum()

np.int64(113500)

### After the JOIN

In [31]:
joined["OrderAmount"].sum()

np.int64(122600)

In [32]:
print("Before JOIN:", orders["OrderAmount"].sum())
print("After JOIN :", joined["OrderAmount"].sum())
print("Difference :", joined["OrderAmount"].sum() - orders["OrderAmount"].sum())

Before JOIN: 113500
After JOIN : 122600
Difference : 9100


### Q10. Are the two sales totals equal?
**Answer:** No. The total before the JOIN is **KSh 113,500** and after the JOIN it is **KSh 122,600**, an overstatement of **KSh 9,100**.

### Q11. If the total changed, which customer caused the duplication?
**Answer:** Customer **`C004` (David)**. His single order `O005` (KSh 9,100) was counted twice because `C004` is duplicated in the customer table. The difference (9,100) equals exactly that order's amount.

### Q12. Why is total reconciliation an important validation step after a JOIN?
**Answer:** A JOIN can run with no error while silently duplicating or dropping rows. Comparing row counts and control totals before and after integration reveals these errors immediately. It shows that the data was transformed faithfully, protects management from decisions based on inflated or understated figures, and often points straight to the underlying key or grain problem.

> **Analytical discipline:** Always compare row counts and important control totals before and after joins.

---
# Part I: Correct the Duplicate-Key Problem
Create a customer table with one record per `CustomerID`, then repeat the JOIN.

> The two `C004` rows were identical, so keeping the first is safe here. In real projects, first check whether duplicate rows actually conflict (different segments or regions) before choosing which to keep.

In [33]:
customers_clean = (
    customers
    .drop_duplicates(subset="CustomerID")
)
customers_clean

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR
10,C010,Jane,SME,NRB


In [34]:
customers_clean.to_sql("customers_clean", conn, if_exists="replace", index=False)

10

In [35]:
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query, conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [36]:
print(len(orders), len(clean_joined))
print(orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum())

15 15
113500 113500


### Q13. Has the corrected JOIN preserved both the row count and the total sales value? Explain.
**Answer:** Yes. Both `orders` and `clean_joined` have **15 rows**, and both totals equal **KSh 113,500**. With `CustomerID` unique in `customers_clean`, each order matches at most one customer, so no order is duplicated. The unmatched order `O015` (`C999`) is still present with NULL customer details, which is correct because it is real revenue that still needs investigating.

---
# Part J: Customer Segment KPIs `[2 Marks]`
Use the cleaned customer table to calculate, for each Segment: number of orders, total sales and average order value.

> **SQL requirement:** the query must use `COUNT()`, `SUM()`, `AVG()` and `GROUP BY`.

In [37]:
query = """
SELECT
    COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
    COUNT(*)                                AS Orders,
    SUM(o.OrderAmount)                      AS TotalSales,
    ROUND(AVG(o.OrderAmount), 2)            AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID
GROUP BY COALESCE(c.Segment, 'Unmatched / NULL')
ORDER BY TotalSales DESC;
"""
segment_kpis = pd.read_sql_query(query, conn)
segment_kpis

,Segment,Orders,TotalSales,AverageOrderValue
0,Corporate,4,46800,11700.00
1,Retail,6,29600,4933.33
2,SME,4,27200,6800.00
3,Unmatched / NULL,1,9900,9900.00


In [38]:
# Control check: segment totals must add back to the original total
print(segment_kpis["Orders"].sum(), segment_kpis["TotalSales"].sum())

15 113500


| Segment | Orders | Total Sales (KSh) | Average Order Value (KSh) |
|---|---:|---:|---:|
| Corporate | 4 | 46,800 | 11,700.00 |
| Retail | 6 | 29,600 | 4,933.33 |
| SME | 4 | 27,200 | 6,800.00 |
| Unmatched / NULL | 1 | 9,900 | 9,900.00 |

### Q14. Write three sentences explaining what management can learn from the segment KPIs.
**Answer:** Corporate customers generate the most revenue (KSh 46,800, about 41% of sales) from just 4 orders, with a very high average order value of KSh 11,700. Retail has the most orders (6) but the lowest average order value (KSh 4,933.33), so it contributes volume but less value per order. A further KSh 9,900 (about 9%) cannot be attributed to any segment because of the unmatched customer `C999`, so fixing that data gap is needed before the segment picture is complete.

---
# Part K: Merge the JSON Region Lookup
The region names are stored separately in JSON. Load this source and merge it with the cleaned customer table.

In [39]:
regions_df = pd.read_json("regions.json")
regions_df

,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [40]:
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


In [41]:
# Confirm every customer received a RegionName
print("Customers missing RegionName:", customers_regions["RegionName"].isna().sum())
customers_regions[["CustomerID", "CustomerName", "RegionCode", "RegionName"]]

Customers missing RegionName: 0


,CustomerID,CustomerName,RegionCode,RegionName
0,C001,Amina,NRB,Nairobi
1,C002,Brian,MSA,Mombasa
2,C003,Carol,NRB,Nairobi
3,C004,David,KSM,Kisumu
4,C005,Esther,NKR,Nakuru
5,C006,Faith,NRB,Nairobi
6,C007,George,MSA,Mombasa
7,C008,Halima,KSM,Kisumu
8,C009,Ian,NKR,Nakuru
9,C010,Jane,NRB,Nairobi


### Q15. Confirm that each customer now has a readable `RegionName` rather than only a `RegionCode`.
**Answer:** Confirmed. All 10 unique customers have a `RegionName` (0 missing): `NRB` is Nairobi, `MSA` is Mombasa, `KSM` is Kisumu and `NKR` is Nakuru.

### Q16. If a `RegionCode` did not exist in `regions.json`, what would a left merge produce for `RegionName`?
**Answer:** The customer row would still be kept (a left merge preserves every row on the left), but `RegionName` would be **`NaN`** (missing / NULL) for that row. This shows up as an unmatched lookup that should be investigated, for example by checking `isna().sum()` as above.

---
# Final Challenge: Regional Sales Analysis
Using SQL and/or pandas, produce a table showing the number of orders and total sales by region.



In [42]:
query = """
SELECT
    o.OrderID,
    o.OrderAmount,
    c.RegionCode
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""
order_regions = pd.read_sql_query(query, conn)

order_regions = order_regions.merge(regions_df, on="RegionCode", how="left")
order_regions["RegionName"] = order_regions["RegionName"].fillna("Unmatched / NULL")

regional_sales = (
    order_regions
    .groupby("RegionName", as_index=False)
    .agg(NumberOfOrders=("OrderID", "count"), TotalSales=("OrderAmount", "sum"))
    .sort_values("TotalSales", ascending=False)
    .reset_index(drop=True)
)
regional_sales

,RegionName,NumberOfOrders,TotalSales
0,Nairobi,7,39800
1,Mombasa,3,23300
2,Nakuru,2,20400
3,Kisumu,2,20100
4,Unmatched / NULL,1,9900


In [43]:
# Control check
print("Orders:", regional_sales["NumberOfOrders"].sum(), "| Total sales:", regional_sales["TotalSales"].sum())

Orders: 15 | Total sales: 113500


| Region | Number of Orders | Total Sales (KSh) |
|---|---:|---:|
| Nairobi | 7 | 39,800 |
| Mombasa | 3 | 23,300 |
| Nakuru | 2 | 20,400 |
| Kisumu | 2 | 20,100 |
| *Unmatched / NULL* | *1* | *9,900* |
| **Total** | **15** | **113,500** |

### Q17. Which region generated the highest sales?
**Answer:** **Nairobi**, with KSh 39,800 from 7 orders (about 35% of total sales). Mombasa is second with KSh 23,300.

### Q18. Should management automatically conclude that this is the "best-performing" region? Explain using at least two limitations of the available data.
**Answer:** No. Highest revenue is not the same as best performance. Limitations of this data:
1. **No profit or cost information.** We only have revenue, so Nairobi may have lower margins or a higher cost to serve than other regions.
2. **No market-size or customer-quality context.** Nairobi has 4 of the 10 customers (and 7 of the 15 orders), so it may simply have more customers; sales per customer or per market size could tell a different story. We also cannot judge customer loyalty or repeat business.
3. **Very small sample and a data gap.** There are only 15 orders over an unknown period, so one large order can change the ranking (Mombasa, Nakuru and Kisumu are within about KSh 3,200 of each other), and KSh 9,900 of sales is unattributed because of `C999`.

---
# Short Conclusion
The biggest data-quality problem was the duplicated customer key `C004`, which made the customer table non-unique at customer grain (a separate issue was the unmatched order customer `C999`). It affected the JOIN by duplicating order `O005`, raising the row count from 15 to 16 and inflating total sales from KSh 113,500 to KSh 122,600. Totals must be checked after joining because a JOIN can run without error while silently duplicating or dropping records. The Corporate segment generated the most sales (KSh 46,800), ahead of Retail (KSh 29,600) and SME (KSh 27,200). A key limitation is that the analysis uses revenue only on a tiny dataset, with KSh 9,900 of sales unattributed to any customer, so it says nothing about profitability or long-term performance.

In [1]:
conn.close()

NameError: name 'conn' is not defined